In [2]:
import duckdb, sys
print(duckdb.__version__, sys.executable)

2.0.0 c:\Users\Melckzedek Kaisha\Desktop\Projects\Urban Taxi Forcasting\nyc-taxi-demand\.venv\Scripts\python.exe


1.load

In [3]:
import duckdb

con = duckdb.connect("../taxi.duckdb")
con.execute("""
    CREATE OR REPLACE TABLE trips AS
    SELECT * FROM read_parquet('../data/raw/yellow_tripdata_2024-*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2: row count

In [7]:
con.execute("SELECT COUNT(*) FROM trips").fetchone()

(9554778,)

3: structure

In [8]:
con.execute("DESCRIBE trips").fetchdf()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


4: date range

In [6]:
con.execute("""
    SELECT MIN(tpep_pickup_datetime) AS earliest,
           MAX(tpep_pickup_datetime) AS latest
    FROM trips
""").fetchdf()

,earliest,latest
0,2002-12-31 22:17:10,2024-04-01 00:34:55


In [9]:
#5: how many rows fall outside Jan-Mar 2024?
con.execute("""
    SELECT COUNT(*) AS out_of_range
    FROM trips
    WHERE tpep_pickup_datetime < '2024-01-01'
       OR tpep_pickup_datetime >= '2024-04-01'
""").fetchdf()

,out_of_range
0,21


In [10]:
# Cell 6: rows per month
con.execute("""
    SELECT date_trunc('month', tpep_pickup_datetime) AS month,
           COUNT(*) AS trips
    FROM trips
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

,month,trips
0,2002-12-01,4
1,2008-12-01,1
2,2009-01-01,4
3,2023-12-01,10
4,2024-01-01,2964617
5,2024-02-01,3007533
6,2024-03-01,3582607
7,2024-04-01,2


In [11]:
# Cell 7: look at the offenders
con.execute("""
    SELECT tpep_pickup_datetime, tpep_dropoff_datetime,
           trip_distance, total_amount
    FROM trips
    WHERE tpep_pickup_datetime < '2024-01-01'
       OR tpep_pickup_datetime >= '2024-04-01'
    ORDER BY tpep_pickup_datetime
    LIMIT 20
""").fetchdf()

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,total_amount
0,2002-12-31 22:17:10,2002-12-31 22:42:24,1.40,18.00
1,2002-12-31 22:59:39,2002-12-31 23:05:41,0.63,-10.50
2,2002-12-31 22:59:39,2002-12-31 23:05:41,0.63,10.50
3,2002-12-31 23:08:30,2003-01-01 14:58:35,5.34,25.15
4,2008-12-31 22:52:49,2008-12-31 23:04:09,1.62,19.90
5,2009-01-01 00:02:13,2009-01-01 00:48:28,0.57,17.16
6,2009-01-01 00:24:09,2009-01-01 01:13:00,10.88,68.29
7,2009-01-01 23:30:39,2009-01-02 00:01:39,10.99,50.00
8,2009-01-01 23:58:40,2009-01-02 00:01:40,0.46,9.40
9,2023-12-31 23:39:17,2023-12-31 23:42:00,0.47,10.10
